In [4]:
#AI Assignment 2
#Muhammad Ali Farooq
#24I-2576

In [24]:
import random
import sys
sys.setrecursionlimit(500)

In [6]:
COLORS = ['Red', 'Blue', 'Green', 'Yellow']
 
class Card:
    def __init__(self, color, number):
        self.color = color
        self.number = number
 
    @property
    def is_skip(self):
        return self.number == 'Skip'
 
    def __str__(self):
        return f"{self.color} {self.number}"
 
    def clone(self):
        return Card(self.color, self.number)

In [7]:
def build_deck():
    deck = []
    for c in COLORS:
        for n in range(10):
            deck.append(Card(c, n))
        deck.append(Card(c, 'Skip'))
        deck.append(Card(c, 'Skip'))
    random.shuffle(deck)
    return deck

In [8]:
def get_valid_moves(hand, top_card):
    valid = []
    for c in hand:
        if c.color == top_card.color:
            valid.append(c)
        elif c.is_skip and top_card.is_skip:
            valid.append(c)
        elif not c.is_skip and not top_card.is_skip and c.number == top_card.number:
            valid.append(c)
    return valid

In [9]:
def init_state():
    deck = build_deck()
    hands = [[], [], []]
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop())
    top_card = None
    while True:
        top_card = deck.pop()
        if not top_card.is_skip:
            break
    return {
        'hands': hands,
        'top_card': top_card,
        'deck': deck,
        'current_player': 0,
        'game_over': False,
        'winner': None
    }

In [10]:
def clone_state(s):
    return {
        'hands': [[c.clone() for c in h] for h in s['hands']],
        'top_card': s['top_card'].clone(),
        'deck': [c.clone() for c in s['deck']],
        'current_player': s['current_player'],
        'game_over': s['game_over'],
        'winner': s['winner']
    }

In [11]:
def apply_move(state, player_idx, move):
    s = clone_state(state)
    if move is None:
        if len(s['deck']) == 0:
            s['deck'] = build_deck()
        s['hands'][player_idx].append(s['deck'].pop())
        s['current_player'] = (player_idx + 1) % 3
    else:
        hand = s['hands'][player_idx]
        for i, c in enumerate(hand):
            if c.color == move.color and c.number == move.number:
                hand.pop(i)
                break
        s['top_card'] = move.clone()
        if not s['hands'][player_idx]:
            s['game_over'] = True
            s['winner'] = player_idx
        if move.is_skip:
            s['current_player'] = (player_idx + 2) % 3
        else:
            s['current_player'] = (player_idx + 1) % 3
    return s

In [12]:
def evaluate(state, player_idx, offensive=False):
    cai = len(state['hands'][player_idx])
    others = [i for i in range(3) if i != player_idx]
    copp = (len(state['hands'][others[0]]) + len(state['hands'][others[1]])) / 2
    skips = sum(1 for c in state['hands'][player_idx] if c.is_skip)
    if offensive:
        return 50 - 7 * cai + 3 * copp + 2 * skips
    else:
        return 50 - 5 * cai + 2 * copp + 4 * skips

In [13]:
tree_lines = []

def log_tree(indent, label):
    if len(tree_lines) < 60:
        tree_lines.append("  " * min(indent, 6) + label)

In [14]:
def minimax(state, depth, player_idx, is_max, alpha, beta, log=False, indent=0):
    if state['game_over'] or depth == 0:
        v = evaluate(state, player_idx, offensive=False)
        if log:
            log_tree(indent, f"[LEAF] eval={v:.1f}")
        return v
 
    cur = state['current_player']
    moves = get_valid_moves(state['hands'][cur], state['top_card'])
    all_moves = (moves + [None])[:4]  # cap breadth
 
    if is_max:
        best = float('-inf')
        for m in all_moves:
            if log:
                log_tree(indent, f"[MAX] P{cur+1} -> {m if m else 'Draw'}")
            ns = apply_move(state, cur, m)
            v = minimax(ns, depth - 1, player_idx,
                        ns['current_player'] == player_idx,
                        alpha, beta, log, indent + 1)
            best = max(best, v)
            alpha = max(alpha, v)
            if beta <= alpha:
                if log:
                    log_tree(indent, "[PRUNE]")
                break
        return best
    else:
        worst = float('inf')
        for m in all_moves[:3]:
            if log:
                log_tree(indent, f"[MIN] P{cur+1} -> {m if m else 'Draw'}")
            ns = apply_move(state, cur, m)
            v = minimax(ns, depth - 1, player_idx,
                        ns['current_player'] == player_idx,
                        alpha, beta, log, indent + 1)
            worst = min(worst, v)
            beta = min(beta, v)
            if beta <= alpha:
                break
        return worst

In [15]:
def minimax_best_move(state, player_idx, log=False):
    global tree_lines
    tree_lines = []
    moves = get_valid_moves(state['hands'][player_idx], state['top_card'])
    all_moves = moves + [None]
    if log:
        log_tree(0, f"=== P{player_idx+1} MINIMAX (Defensive) Depth=3 ===")
        log_tree(0, f"Top card: {state['top_card']}")
        log_tree(0, f"Hand: {[str(c) for c in state['hands'][player_idx]]}")
        log_tree(0, f"Valid plays: {[str(m) for m in moves]} + [Draw]")
        log_tree(0, "")
 
    best_move, best_val = None, float('-inf')
    scores = []
    for m in all_moves:
        if log:
            log_tree(0, f"[ROOT] {m if m else 'Draw'}")
        ns = apply_move(state, player_idx, m)
        v = minimax(ns, 2, player_idx, ns['current_player'] == player_idx,
                    float('-inf'), float('inf'), log, 1)
        scores.append((m, v))
        if log:
            log_tree(0, f"  => score={v:.1f}\n")
        if v > best_val:
            best_val = v
            best_move = m
    if log:
        log_tree(0, f"BEST -> {best_move if best_move else 'Draw'} (score={best_val:.1f})")
    return best_move, best_val, scores

In [16]:
def expectimax(state, depth, player_idx, log=False, indent=0):
    if state['game_over'] or depth == 0:
        v = evaluate(state, player_idx, offensive=True)
        if log:
            log_tree(indent, f"[LEAF] eval={v:.1f}")
        return v
 
    cur = state['current_player']
    moves = get_valid_moves(state['hands'][cur], state['top_card'])
 
    if cur == player_idx:
        # MAX node — try all plays; Draw goes to CHANCE node
        all_moves = (moves + [None])[:4]
        best = float('-inf')
        for m in all_moves:
            if m is None:
                if log:
                    log_tree(indent, f"[MAX] P{cur+1} -> Draw => CHANCE")
                v = chance_node_ev(state, player_idx, depth - 1, log, indent + 1)
            else:
                if log:
                    log_tree(indent, f"[MAX] P{cur+1} -> {m}")
                ns = apply_move(state, cur, m)
                v = expectimax(ns, depth - 1, player_idx, log, indent + 1)
            best = max(best, v)
        return best
    else:
        # Opponent node — pick best of a small sample (not adversarial, stochastic)
        all_moves = (moves + [None])[:2]
        if not all_moves:
            return evaluate(state, player_idx, offensive=True)
        m = all_moves[0]   # deterministic: pick first valid (simulate one opponent move)
        if log:
            log_tree(indent, f"[OPP] P{cur+1} -> {m if m else 'Draw'}")
        ns = apply_move(state, cur, m)
        return expectimax(ns, depth - 1, player_idx, log, indent + 1)

In [17]:
def chance_node_ev(state, player_idx, depth, log=False, indent=0):
    """Expected value over 4 sampled deck cards."""
    if not state['deck']:
        return evaluate(state, player_idx, offensive=True)
    sample = state['deck'][-min(len(state['deck']), 4):]
    p = 1.0 / len(sample)
    ev = 0.0
    for drawn in sample:
        ns = clone_state(state)
        ns['hands'][player_idx].append(drawn.clone())
        ns['current_player'] = (state['current_player'] + 1) % 3
        if log:
            log_tree(indent, f"[CHANCE] p={p:.2f} draw {drawn}")
        ev += p * evaluate(ns, player_idx, offensive=True)   # leaf eval at chance level
    return ev

In [18]:
def expectimax_best_move(state, player_idx, log=False):
    global tree_lines
    tree_lines = []
    moves = get_valid_moves(state['hands'][player_idx], state['top_card'])
    all_moves = moves + [None]
    if log:
        log_tree(0, f"=== P{player_idx+1} EXPECTIMAX (Offensive) Depth=3 ===")
        log_tree(0, f"Top card: {state['top_card']}")
        log_tree(0, f"Hand: {[str(c) for c in state['hands'][player_idx]]}")
        log_tree(0, f"Valid plays: {[str(m) for m in moves]} + [Draw]")
        log_tree(0, "")
 
    best_move, best_val = None, float('-inf')
    scores = []
    for m in all_moves:
        if m is None:
            if log:
                log_tree(0, "[ROOT] Draw => CHANCE NODE")
            v = chance_node_ev(state, player_idx, 2, log, 1)
        else:
            if log:
                log_tree(0, f"[ROOT] {m}")
            ns = apply_move(state, player_idx, m)
            v = expectimax(ns, 2, player_idx, log, 1)
        scores.append((m, v))
        if log:
            log_tree(0, f"  => EV={v:.2f}\n")
        if v > best_val:
            best_val = v
            best_move = m
    if log:
        log_tree(0, f"BEST -> {best_move if best_move else 'Draw'} (EV={best_val:.2f})")
    return best_move, best_val, scores

In [19]:
SEP = '─' * 65
 
def print_state(state, turn_num):
    print('═' * 65)
    print(f"  TURN {turn_num}")
    print(SEP)
    print(f"  Top Card  : {state['top_card']}")
    print(f"  Deck Size : {len(state['deck'])} cards remaining")
    print(SEP)
    names = [
        'P1 (Minimax   / Defensive)',
        'P2 (Expectimax / Offensive)',
        'P3 (Minimax   / Simulation)',
    ]
    for i, (name, hand) in enumerate(zip(names, state['hands'])):
        mark = ">>>" if state['current_player'] == i else "   "
        print(f"  {mark} {name}: {len(hand)} cards | {[str(c) for c in hand]}")
    print(SEP)
    print(f"  Eval Scores ->  "
          f"P1: {evaluate(state,0,False):.1f}  |  "
          f"P2: {evaluate(state,1,True):.1f}  |  "
          f"P3: {evaluate(state,2,False):.1f}")
 
 
def print_decision(pname, scores, best_move, best_val, algo):
    print(f"\n  [{pname}] {algo} — All candidates (depth=3):")
    for m, v in scores:
        tag = " ◄ BEST" if m == best_move or (m is None and best_move is None) else ""
        print(f"      Play: {str(m) if m else 'Draw':<22}  Score: {v:.2f}{tag}")
    print(f"\n  Decision -> {str(best_move) if best_move else 'Draw (draw 1 card)'}  "
          f"[score={best_val:.2f}]")
 
 
def print_tree(lines, max_lines=28):
    print(f"\n  {'─'*18} SEARCH TREE (excerpt) {'─'*18}")
    for line in lines[:max_lines]:
        print("  " + line)
    if len(lines) > max_lines:
        print(f"  ... ({len(lines) - max_lines} more nodes not shown)")
    print(f"  {'─'*60}")

In [26]:
def run_game():
    random.seed(98)
    state = init_state()
    turn_num = 0
    SHOW_TREE = {1, 2, 3}
 
    print()
    print("╔" + "═"*63 + "╗")
    print("║" + "  UNO AI — Adversarial Search Simulation".center(63) + "║")
    print("║" + "  P1: Minimax (Defensive)  |  P2: Expectimax (Offensive)".center(63) + "║")
    print("║" + "  P3: Minimax (Simulation)".center(63) + "║")
    print("╚" + "═"*63 + "╝")
    print()
    print("  EVALUATION FORMULA:")
    print("    Defensive : Score = 50 - 5·C_AI + 2·C_opp + 4·S")
    print("    Offensive : Score = 50 - 7·C_AI + 3·C_opp + 2·S")
    print()
 
    while not state['game_over']:
        turn_num += 1
        print_state(state, turn_num)
 
        p = state['current_player']
        show = turn_num in SHOW_TREE
 
        if p == 0:
            best_move, best_val, scores = minimax_best_move(state, 0, log=show)
            print_decision("P1", scores, best_move, best_val, "Minimax (Defensive)")
        elif p == 1:
            best_move, best_val, scores = expectimax_best_move(state, 1, log=show)
            print_decision("P2", scores, best_move, best_val, "Expectimax (Offensive)")
        else:
            best_move, best_val, scores = minimax_best_move(state, 2, log=show)
            print_decision("P3", scores, best_move, best_val, "Minimax (Simulation)")
 
        if show and tree_lines:
            print_tree(tree_lines)
 
        if best_move is None:
            drawn = state['deck'][-1] if state['deck'] else None
            state = apply_move(state, p, None)
            print(f"\n  >>> P{p+1} draws a card{f': {drawn}' if drawn else ''}.")
        else:
            state = apply_move(state, p, best_move)
            sk = "  *** SKIP — next player's turn is skipped! ***" if best_move.is_skip else ""
            print(f"\n  >>> P{p+1} plays: {best_move}.{sk}")
 
        if state['game_over']:
            break
        if turn_num >= 150:
            print("\n  [Turn limit reached — game ended]")
            break
 
    print()
    print('═' * 65)
    if state['winner'] is not None:
        w = state['winner']
        wnames = ['Player 1 (Minimax/Defensive)',
                  'Player 2 (Expectimax/Offensive)',
                  'Player 3 (Minimax/Simulation)']
        print(f"\n WINNER: {wnames[w]}  —  finished in {turn_num} turns!\n")
        print("  Final hand sizes:")
        for i, hand in enumerate(state['hands']):
            print(f"    P{i+1}: {len(hand)} cards  {[str(c) for c in hand]}")
    print()
    print('═' * 65)

In [27]:
run_game()


╔═══════════════════════════════════════════════════════════════╗
║              UNO AI — Adversarial Search Simulation           ║
║      P1: Minimax (Defensive)  |  P2: Expectimax (Offensive)   ║
║                     P3: Minimax (Simulation)                  ║
╚═══════════════════════════════════════════════════════════════╝

  EVALUATION FORMULA:
    Defensive : Score = 50 - 5·C_AI + 2·C_opp + 4·S
    Offensive : Score = 50 - 7·C_AI + 3·C_opp + 2·S

═════════════════════════════════════════════════════════════════
  TURN 1
─────────────────────────────────────────────────────────────────
  Top Card  : Red 0
  Deck Size : 32 cards remaining
─────────────────────────────────────────────────────────────────
  >>> P1 (Minimax   / Defensive): 5 cards | ['Blue Skip', 'Yellow Skip', 'Green 6', 'Red 4', 'Green 1']
      P2 (Expectimax / Offensive): 5 cards | ['Yellow 0', 'Green 2', 'Blue 6', 'Green 7', 'Yellow 9']
      P3 (Minimax   / Simulation): 5 cards | ['Red 2', 'Red 3', 'Green Skip